# 03 — Merging FBref and Transfermarkt

**Goal:** Link FBref performance data to Transfermarkt valuations using fuzzy player name matching. FBref and Transfermarkt share no common player ID so we match on name + club.

**Input:** `data/cleaned/fbref_cleaned.csv`, `data/cleaned/transfermarkt_cleaned.csv`  
**Output:** `data/final/merged.csv`

**Key challenges documented:**
- Accented characters (Müller vs Muller)
- Name format differences (Last, First vs First Last)
- Club name mismatches (Bayern Munich vs FC Bayern München)
- Mid-season transfers (player at different clubs in each dataset)

## 1. Imports and Config

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from fuzzywuzzy import fuzz, process
from unidecode import unidecode

CLEANED_DIR = Path('../data/cleaned')
FINAL_DIR   = Path('../data/final')
FINAL_DIR.mkdir(parents=True, exist_ok=True)

# Minimum fuzzy match score to accept a match (0-100)
# 85 is conservative — keeps precision high, some true matches may be lost
FUZZY_THRESHOLD = 85

## 2. Load Cleaned Data

In [ ]:
fbref = pd.read_csv(CLEANED_DIR / 'fbref_cleaned.csv')
tm    = pd.read_csv(CLEANED_DIR / 'transfermarkt_cleaned.csv')

print(f'FBref:          {fbref.shape}')
print(f'Transfermarkt:  {tm.shape}')
print(f'\nFBref leagues:  {fbref["league"].value_counts().to_dict()}')
print(f'TM leagues:     {tm["league"].value_counts().to_dict()}')

## 3. Normalise Names for Matching

Strip accents, lowercase, remove punctuation. This closes most encoding gaps between sources.

In [ ]:
def normalise_name(name: str) -> str:
    """Lowercase, strip accents, remove non-alpha characters."""
    if pd.isna(name):
        return ''
    return unidecode(str(name)).lower().strip()

fbref['name_norm'] = fbref['player_name'].apply(normalise_name)
tm['name_norm']    = tm['player_name'].apply(normalise_name)

# Preview
print('FBref sample names:')
print(fbref[['player_name', 'name_norm']].head(5).to_string())
print('\nTM sample names:')
print(tm[['player_name', 'name_norm']].head(5).to_string())

## 4. Exact Match Pass

First try exact match on normalised name within the same league — fast and zero false positives.

In [ ]:
merged_exact = fbref.merge(
    tm[['player_id', 'name_norm', 'league', 'market_value_eur', 'market_value_m',
        'highest_market_value_in_eur', 'age', 'nationality', 'position_group', 'sub_position']],
    on=['name_norm', 'league'],
    how='inner'
)
merged_exact['match_type'] = 'exact'

print(f'Exact matches: {len(merged_exact)}')

# Unmatched FBref players
fbref_unmatched = fbref[~fbref['name_norm'].isin(merged_exact['name_norm'])].copy()
print(f'FBref players still unmatched: {len(fbref_unmatched)}')

## 5. Fuzzy Match Pass

For unmatched players, run fuzzy matching within each league. Uses `token_sort_ratio` which handles word-order differences (e.g. "Silva Bruno" vs "Bruno Silva").

**Note:** This cell takes 1-3 minutes to run.

In [ ]:
def fuzzy_match_league(fbref_sub: pd.DataFrame, tm_sub: pd.DataFrame,
                        threshold: int = FUZZY_THRESHOLD) -> pd.DataFrame:
    """Fuzzy match unmatched FBref players against TM players within same league."""
    tm_names = tm_sub['name_norm'].tolist()
    tm_lookup = tm_sub.set_index('name_norm')

    rows = []
    for _, row in fbref_sub.iterrows():
        match = process.extractOne(
            row['name_norm'], tm_names,
            scorer=fuzz.token_sort_ratio
        )
        if match and match[1] >= threshold:
            tm_row = tm_lookup.loc[match[0]]
            # If multiple TM players have the same normalised name, take the first
            if isinstance(tm_row, pd.DataFrame):
                tm_row = tm_row.iloc[0]
            result = row.to_dict()
            result.update({
                'player_id':                  tm_row.get('player_id'),
                'market_value_eur':            tm_row.get('market_value_eur'),
                'market_value_m':              tm_row.get('market_value_m'),
                'highest_market_value_in_eur': tm_row.get('highest_market_value_in_eur'),
                'nationality':                 tm_row.get('nationality'),
                'position_group':              tm_row.get('position_group'),
                'sub_position':                tm_row.get('sub_position'),
                'tm_name_matched':             match[0],
                'fuzzy_score':                 match[1],
                'match_type':                  'fuzzy',
            })
            rows.append(result)
    return pd.DataFrame(rows)


fuzzy_frames = []
for league in fbref_unmatched['league'].unique():
    fb_sub = fbref_unmatched[fbref_unmatched['league'] == league]
    tm_sub = tm[tm['league'] == league]
    # Exclude already exact-matched TM names
    matched_names = set(merged_exact[merged_exact['league'] == league]['name_norm'])
    tm_sub = tm_sub[~tm_sub['name_norm'].isin(matched_names)]

    result = fuzzy_match_league(fb_sub, tm_sub)
    print(f'{league:25} unmatched: {len(fb_sub):>4} → fuzzy matched: {len(result):>4}')
    fuzzy_frames.append(result)

merged_fuzzy = pd.concat(fuzzy_frames, ignore_index=True) if fuzzy_frames else pd.DataFrame()

## 6. Combine and Review

In [ ]:
merged = pd.concat([merged_exact, merged_fuzzy], ignore_index=True)

print(f'Total matched players: {len(merged)}')
print(f'  Exact:  {len(merged_exact)}')
print(f'  Fuzzy:  {len(merged_fuzzy)}')
print(f'\nMatch rate: {len(merged)/len(fbref)*100:.1f}% of FBref players matched')
print(f'\nBy league:')
print(merged['league'].value_counts())

## 7. Spot-Check Fuzzy Matches

Review low-confidence matches — these are the ones most likely to be wrong.

In [ ]:
if 'fuzzy_score' in merged.columns:
    low_confidence = merged[merged['fuzzy_score'] < 92].sort_values('fuzzy_score')
    print(f'Low-confidence fuzzy matches (score < 92): {len(low_confidence)}')
    if len(low_confidence):
        print(low_confidence[['player_name', 'tm_name_matched', 'fuzzy_score', 'club', 'league']]
              .head(20).to_string())

## 8. Drop Duplicates and Unvaluable Players

In [ ]:
# If a player matched multiple TM records, keep the higher fuzzy score
if 'fuzzy_score' in merged.columns:
    merged = merged.sort_values('fuzzy_score', ascending=False)
merged = merged.drop_duplicates(subset=['player_name', 'league'], keep='first')

# Drop players with no market value — required for scoring
before = len(merged)
merged = merged[merged['market_value_eur'].notna() & (merged['market_value_eur'] > 0)]
print(f'Dropped {before - len(merged)} players with no market value')
print(f'Final merged dataset: {len(merged)} players')

## 9. Save

In [ ]:
out = FINAL_DIR / 'merged.csv'
merged.to_csv(out, index=False)
print(f'Saved {len(merged)} players, {merged.shape[1]} columns to {out}')
print(f'\nColumns: {list(merged.columns)}')